In [1]:
# ProjectTwoDashboard.ipynb
from dash import Dash, dcc, html, dash_table
import dash_leaflet as dl
import plotly.express as px
from dash.dependencies import Input, Output
import pandas as pd
import base64
import os
from animal_shelter import AnimalShelter 

# --- 1. Database Connection Logic ---
# Using empty credentials for local MongoDB access
username = ""
password = ""
host = "localhost" 
port = 27017 

# Initialize db as None to prevent NameError in callbacks
db = None

try:
    # This creates the bridge to your local MongoDB service
    db = AnimalShelter(username, password, host, port)
    print("Connected successfully to MongoDB.")
except Exception as e:
    # Generic exception handling avoids the 'pymongo.errors' attribute error
    print(f"Connection Alert: {e}")

# Initial Data Load for the table
df = pd.DataFrame()
if db is not None:
    try:
        data = db.read({})
        df = pd.DataFrame.from_records(data)
        # DROP _id for Dash compatibility
        if not df.empty and '_id' in df.columns:
            df.drop(columns=['_id'], inplace=True)
        print(f"Initial DataFrame loaded with {len(df)} rows.")
    except Exception as e:
        print(f"Error loading initial data: {e}")

# --- 2. Initialize Dash App ---
# Modern Dash handles Jupyter environments automatically
app = Dash(__name__)

# Load Grazioso Salvare Logo
image_filename = 'Grazioso_Salvare_Logo.png'
encoded_image = ""
if os.path.isfile(image_filename):
    with open(image_filename, 'rb') as f:
        encoded_image = base64.b64encode(f.read()).decode('ascii')

# Dashboard Layout
app.layout = html.Div([
    html.Center(html.B(html.H1('CS-340 Dashboard - Grazioso Salvare'))),
    html.Div([
        html.A(
            html.Img(
                src=f'data:image/png;base64,{encoded_image}' if encoded_image else '',
                style={'width': '200px', 'height': 'auto', 'marginRight': '20px'}
            ),
            href="http://www.snhu.edu",
            target="_blank"
        ),
        html.Div(
            html.P("Dashboard Creator: Saad Ohiduzzaman"),
            style={'display': 'inline-block', 'verticalAlign': 'top', 'marginTop': '20px'}
        )
    ], style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center'}),
    html.Hr(),
    html.H2("Filter Dogs by Rescue Type:"),
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'Water Rescue'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain or Wilderness Rescue'},
            {'label': 'Disaster or Individual Tracking', 'value': 'Disaster or Individual Tracking'},
            {'label': 'Reset', 'value': 'Reset'}
        ],
        value='Reset',
        inline=True,
        style={'margin': '10px'}
    ),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i} for i in df.columns if i not in ['location_lat', 'location_long']],
        data=df.to_dict('records'),
        page_size=10,
        filter_action='native',
        sort_action='native',
        row_selectable='single',
        selected_rows=[0] if not df.empty else [], # FIX: Prevents layout crash when DataFrame is empty
    ),
    html.Hr(),
    html.Div([
        dcc.Graph(id='pie-chart-id', style={'width': '49%', 'display': 'inline-block'}),
        html.Div(id='map-id', style={'width': '49%', 'display': 'inline-block'})
    ])
])

# --- 3. Callbacks ---

@app.callback(Output('datatable-id', 'data'), [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    # Check if database is initialized to prevent crashes
    if db is None:
        return []

    query = {}
    if filter_type == 'Water Rescue':
        query = {"breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]}, "sex_upon_outcome": "Intact Female"}
    elif filter_type == 'Mountain or Wilderness Rescue':
        query = {"breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Siberian Husky"]}, "sex_upon_outcome": "Intact Male"}
    elif filter_type == 'Disaster or Individual Tracking':
        query = {"breed": {"$in": ["Doberman Pinscher", "Bloodhound"]}, "sex_upon_outcome": "Intact Male"}
    
    try:
        data = db.read(query)
        filtered_df = pd.DataFrame.from_records(data)
        if not filtered_df.empty and '_id' in filtered_df.columns:
            filtered_df.drop(columns=['_id'], inplace=True)
        return filtered_df.to_dict('records')
    except Exception:
        return []

@app.callback(Output('pie-chart-id', "figure"), [Input('datatable-id', "derived_virtual_data")])
def update_pie_chart(viewData):
    if not viewData: 
        # FIX: Provide a valid empty figure format so Plotly doesn't crash the layout
        return px.pie(names=['No Data'], values=[1], title='No data available')
    dff = pd.DataFrame.from_dict(viewData)
    return px.pie(dff, names='breed', title='Distribution of Breeds')

@app.callback(Output('map-id', "children"), [Input('datatable-id', "derived_virtual_data"), Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    if not viewData or not index: return html.Div("Select a row to view location on map")
    dff = pd.DataFrame.from_dict(viewData)
    
    # FIX: Ensure the selected row actually exists in the current view to prevent IndexError
    if index[0] >= len(dff): 
        return html.Div("Select a valid row to view location on map")
        
    row_data = dff.iloc[index[0]]
    return [
        dl.Map(style={'width': '100%', 'height': '400px'}, center=[row_data['location_lat'], row_data['location_long']], zoom=10, children=[
            dl.TileLayer(),
            dl.Marker(position=[row_data['location_lat'], row_data['location_long']], children=[
                dl.Tooltip(row_data.get('name', 'N/A')),
                dl.Popup(html.H3(f"Breed: {row_data.get('breed', 'N/A')}"))
            ])
        ])
    ]

# Start the server using the modern .run() command
if __name__ == '__main__':
    app.run(port=8052, debug=True)

System: Successfully connected to AAC.animals at localhost:27017.
Connected successfully to MongoDB.
Read Success: Found 10000 matching record(s).
Initial DataFrame loaded with 10000 rows.


Read Success: Found 10000 matching record(s).
